# DantinoX — Colab Quickstart

> *"E quindi uscimmo a riveder le stelle."*

This notebook trains a decoder-only Transformer with **DantinoX** on a HuggingFace dataset, generates text, benchmarks throughput, and optionally pushes the checkpoint to the Hub.

**Runtime:** `Runtime → Change runtime type → T4 GPU` (or A100 if available).

| Step | What happens |
|---|---|
| **1 — Install** | JAX CUDA + DantinoX from PyPI |
| **2 — Check GPU** | Verify the accelerator |
| **3 — Configure** | `ModelConfig` (GQA · MLA · MoE · Sliding-Window) + `TrainingConfig` |
| **4 — LR Finder** | Find the optimal learning rate before training *(optional)* |
| **5 — Train** | `Trainer(paradigm, train_cfg).fit()` — streaming, eval split, early stopping, resume |
| **6 — Generate** | Temperature · top-k · top-p · greedy · streaming · batch |
| **7 — ELF** | Continuous flow-matching diffusion *(optional)* |
| **8 — Hub** | Push/pull checkpoint *(optional)* |
| **9 — LoRA** | Fine-tune with adapters *(optional)* |
| **10 — Benchmark** | Prefill latency + decode throughput |
| **11 — Plots** | Performance figures |

**API in one line:** architecture lives in `ModelConfig` / `ELFConfig`, training hyper-parameters in `TrainingConfig`, and a *paradigm* (`ARParadigm`, `DiscreteParadigm`, `ContinuousParadigm`) ties a model to its loss and sampler.

---
## 1 — Install

We upgrade JAX to the CUDA 12 build (Colab ships an older CPU-only version) and install DantinoX from PyPI.

> ⚠️ **Restart the runtime once after this cell** (`Runtime → Restart session`), then continue from cell 2.

In [ ]:
# Upgrade JAX + Flax to CUDA 12 build — must restart runtime after this cell.
# Required: Colab's default JAX is too old for its current CUDA plugin
# (PJRT_FFI version mismatch → JaxRuntimeError on the first GPU call).
# !pip install -q -U "jax[cuda12]" jaxlib flax

# Install / upgrade DantinoX + optional extras (data, hub, elf, benchmark)
!pip install -q -U "dantinox[data,hub,elf,benchmark]"

print("\n✅ Done — restart the runtime now (Runtime → Restart session), then run from the next cell.")

---
## 2 — Check GPU

In [ ]:
import flax
import jax

print(f"JAX  version: {jax.__version__}")
print(f"Flax version: {flax.__version__}")

In [ ]:
import jax

devices = jax.devices()
print("JAX devices:", devices)

if not any("cuda" in str(d).lower() or "gpu" in str(d).lower() for d in devices):
    print("\n⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU")
else:
    print(f"\n✅ Running on {devices[0]}")

---
## 3 — Configure

`Config` exposes every knob: architecture, positional encoding, attention variant, MoE, tokenizer, dataset, optimiser, scheduler, regularisation, LoRA, and multi-GPU sharding.

The cells below define four ready-to-use configurations:

| Config | Attention | Extra | GPU target |
|---|---|---|---|
| `config_gqa` | GQA (baseline) | — | T4 / A100 |
| `config_mla` | MLA (DeepSeek-style) | latent KV | T4 / A100 |
| `config_moe` | GQA | Mixture-of-Experts MLP | A100 |
| `config_swa` | Sliding-Window | — | T4 |

Pick one and assign it to `config` at the bottom of the section.

In [ ]:
import dantinox as dx

# ── Architecture — what the model IS ─────────────────────────────────────────
model_gqa = dx.ModelConfig(
    # ── Core dimensions ──────────────────────────────────────────────────────
    dim=256,             # model width (dim == n_heads * head_size)
    n_heads=8,           # number of query heads
    head_size=32,        # per-head dimension
    num_blocks=6,        # transformer depth
    max_context=256,     # maximum sequence length
    vocab_size=256,      # char tokenizer: ≥ number of distinct characters

    # ── Attention ────────────────────────────────────────────────────────────
    attention="gqa",     # "mha" | "gqa" | "mla"
    kv_heads=2,          # GQA: 4 query heads share each KV head
    use_flash=True,      # memory-efficient attention kernel

    # ── MLP / norm / positions ───────────────────────────────────────────────
    ffn="mlp",           # "mlp" | "moe"
    use_swiglu=True,     # SwiGLU activation (better than GELU for language)
    expansion=4,         # MLP hidden = dim * expansion
    norm="rmsnorm",      # "rmsnorm" (faster) | "layernorm"
    pos_encoding="rotary",  # RoPE (recommended)

    # ── Regularisation ───────────────────────────────────────────────────────
    dropout=0.1,
    weight_tying=True,
    gradient_checkpointing=True,  # recompute activations on backward (saves VRAM)
)

# ── Training — HOW the model is trained ──────────────────────────────────────
train_cfg = dx.TrainingConfig(
    # Optimiser
    optimizer="adamw",   # "adamw" | "adam" | "lion" | "adafactor" | "muon"
    lr=3e-4,
    lr_schedule="wsd",   # "wsd" | "cosine" | "linear" | "constant"
    warmup_steps=100,
    grad_clip=1.0,
    use_bf16=True,       # bfloat16 params (faster + less VRAM on Ampere/T4)

    # Loop
    epochs=300,
    batch_size=64,
    grad_accum=4,        # effective batch = batch_size * grad_accum = 256
    patience=10,         # early stopping: stop after N epochs w/o val improvement
    val_frac=0.1,        # 10% of tokens held out — best checkpoint by VAL loss
    eval_iters=20,       # batches averaged per val-loss estimate
    seed=42,

    # Dataset — streamed from HuggingFace
    dataset_source="huggingface",
    dataset_name="Daniele/dante-corpus",
    dataset_text_field="text",
    streaming=True,

    # Tokenizer
    tokenizer_type="char",   # "char" (fast) | "bpe" | "t5"
)

paradigm_gqa = dx.ARParadigm(model_gqa)
print("GQA:", paradigm_gqa)

In [ ]:
# ── MLA — Multi-head Latent Attention (DeepSeek-style KV compression) ────────
# KV cache is compressed to a low-rank latent space → much smaller memory
# footprint than GQA at the same parameter count.
model_mla = dx.ModelConfig(
    dim=256, n_heads=8, head_size=32, num_blocks=6, max_context=256,
    vocab_size=256,

    attention="mla",     # enable Multi-head Latent Attention
    down_dim_kv=64,      # latent KV dim (≪ dim saves cache; try 32–128)
    down_dim_q=128,      # latent Q  dim (set to dim for full quality)
    rope_dim=32,         # RoPE applied only to this many dims per head

    use_swiglu=True, norm="rmsnorm", use_flash=True,
    weight_tying=True, dropout=0.1, gradient_checkpointing=True,
)
paradigm_mla = dx.ARParadigm(model_mla)
print("MLA:", paradigm_mla)

In [ ]:
# ── MoE — Mixture of Experts MLP (sparse activation) ─────────────────────────
# Each token activates only top_k of n_experts MLP sub-networks.
# Parameters scale with n_experts but FLOPS stay roughly constant.
# Needs more VRAM — recommended for A100.
model_moe = dx.ModelConfig(
    dim=256, n_heads=8, head_size=32, num_blocks=6, max_context=256,
    vocab_size=256, attention="gqa", kv_heads=2,

    ffn="moe",              # replace dense MLP with mixture-of-experts
    n_experts=4,            # total expert sub-networks
    top_k=2,                # experts activated per token (≤ n_experts)
    moe_balance_coeff=0.1,  # load-balancing loss weight (prevents collapse)
    expansion=4,

    use_swiglu=True, norm="rmsnorm", use_flash=True,
    weight_tying=True, dropout=0.1, gradient_checkpointing=True,
)
paradigm_moe = dx.ARParadigm(model_moe)
print("MoE:", paradigm_moe)

In [ ]:
# ── Sliding-Window Attention — O(n · w) instead of O(n²) ─────────────────────
# Each token attends only to the last `context_window` tokens.
# Enables very long sequences on a T4 without OOM.
model_swa = dx.ModelConfig(
    dim=256, n_heads=8, head_size=32, num_blocks=6, max_context=512,
    vocab_size=256, attention="gqa", kv_heads=2,

    sliding_window=True,  # enable local attention
    context_window=64,    # each token sees the previous 64 tokens
    no_sink=True,         # False keeps the first (sink) token always visible

    use_swiglu=True, norm="rmsnorm", use_flash=True,
    weight_tying=True, dropout=0.1, gradient_checkpointing=True,
)
paradigm_swa = dx.ARParadigm(model_swa)
print("SWA:", paradigm_swa)

In [ ]:
# ── Pick the paradigm to train ────────────────────────────────────────────────
paradigm = paradigm_gqa   # swap to paradigm_mla / paradigm_moe / paradigm_swa

---
## 4 — LR Finder *(optional)*

Sweeps the learning rate exponentially over `num_steps` mini-batches and plots loss vs LR.
Run this before a long training run to confirm the LR in your config is well-placed.

In [ ]:
import warnings
import matplotlib.pyplot as plt

from dantinox import Config
from dantinox.trainer import Trainer as LegacyTrainer

# The LR range test still lives in the legacy engine — bridge the split
# configs into a monolithic Config for it.
legacy_cfg = Config.from_parts(paradigm.config, train_cfg)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    suggested_lr, lrs, losses = LegacyTrainer(legacy_cfg).find_lr(
        num_steps=100,  # more steps → smoother curve
        min_lr=1e-7,
        max_lr=1.0,
        smoothing=0.9,
    )
print(f"Suggested LR: {suggested_lr:.2e}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lrs, losses)
ax.axvline(suggested_lr, color="red", linestyle="--", label=f"suggested {suggested_lr:.2e}")
ax.set_xscale("log"); ax.set_xlabel("LR"); ax.set_ylabel("Smoothed loss")
ax.set_title("LR Range Test"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Optionally update the config with the found LR
# train_cfg.lr = suggested_lr

---
## 5 — Train

`Trainer(paradigm, train_cfg).fit()` streams the dataset, holds out a validation split, trains with gradient accumulation + early stopping, and checkpoints the **best model by validation loss**. The full train state (model + optimizer) is saved each epoch, so `fit(..., resume=True)` continues exactly where a crashed run stopped.

In [ ]:
from dantinox import Trainer

run_dir = Trainer(paradigm, train_cfg).fit(
    # data_source=None,      # omit to stream from HuggingFace (set in train_cfg)
    # run_dir="runs/my_run", # fix the output directory; auto-generated if omitted
    # resume=True,           # continue from the saved train state in run_dir
)

print(f"\nCheckpoint saved to: {run_dir}")

---
## 6 — Generate

`Generator` supports single prompts, batched prompts, and streaming — each with independent sampling controls.

In [ ]:
from dantinox import Generator

gen = Generator(
    run_dir,
    seed=42,      # controls sampling randomness; change for different outputs
)

prompt = "Nel mezzo del cammin "

# ── Sampling strategies ───────────────────────────────────────────────────────

# 1. Temperature sampling — higher = more creative, lower = more focused
print("=== temperature=0.8, top_k=40 ===")
print(gen.generate(prompt, max_new_tokens=120, temperature=0.8, top_k=40))

# 2. Nucleus (top-p) sampling — keeps the smallest set of tokens whose
#    cumulative probability ≥ top_p; adapts vocabulary size dynamically
print("\n=== top_p=0.9 ===")
print(gen.generate(prompt, max_new_tokens=120, temperature=1.0, top_p=0.9))

# 3. Greedy decoding — always picks the most likely token (deterministic)
print("\n=== greedy ===")
print(gen.generate(prompt, max_new_tokens=120, greedy=True))

# 4. Combined top-k + top-p + temperature (common production setting)
print("\n=== top_k=50, top_p=0.95, temperature=0.7 ===")
print(gen.generate(prompt, max_new_tokens=200, top_k=50, top_p=0.95, temperature=0.7))

In [ ]:
# ── Batch generation — one forward pass ──────────────────────────
prompts = [
    "Nel mezzo del cammin ",
    "Lasciate ogni speranza ",
    "Per me si va nella città dolente ",
]

outputs = gen.generate_batch(prompts, max_new_tokens=80, temperature=0.9)
for p, o in zip(prompts, outputs):
    print(f"▶ {p!r}")
    print(o)
    print()

In [ ]:
# ── Streaming — tokens printed as they arrive ─────────────────────
print("Streaming: ", end="", flush=True)
for chunk in gen.stream("Nel mezzo del cammin ", max_new_tokens=150, temperature=0.8):
    print(chunk, end="", flush=True)
print()

---
## 7 — ELF: continuous flow-matching *(optional)*

**ELF (Embedded Language Flows)** generates text by denoising in a continuous embedding space instead of sampling tokens autoregressively. It needs:

- the `transformers` extra (`pip install dantinox[elf]`) — a **frozen T5 encoder** provides the contextual embedding space
- `tokenizer_type="t5"` and `vocab_size=32_128` (the T5 vocabulary)
- `embed_dim` matching the T5 variant: `t5-small → 512`, `t5-base → 768`

Architecture goes in `ELFConfig`, training hyper-parameters in `TrainingConfig`, exactly like the AR case.

In [ ]:
# A small local corpus for the ELF demo
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O tiny_shakespeare.txt
!head -c 300 tiny_shakespeare.txt

In [ ]:
import dantinox as dx

elf_cfg = dx.ELFConfig(
    # ── Embedding / flow space ───────────────────────────────────────────────
    embed_dim=512,        # must match the frozen T5 encoder (t5-small → 512)
    bottleneck_dim=64,    # bottleneck between embed and model space

    # ── Transformer backbone ─────────────────────────────────────────────────
    model_dim=256,        # transformer width (= n_heads × head_size)
    n_heads=4,
    head_size=64,
    num_blocks=4,
    vocab_size=32_128,    # T5 vocabulary (fixed)
    max_seq_len=128,

    # ── Inference ────────────────────────────────────────────────────────────
    elf_n_steps=32,       # denoising steps at generation time
    elf_cfg_scale=1.5,    # classifier-free-guidance scale

    t5_model_name="t5-small",
)
elf_paradigm = dx.ContinuousParadigm(elf_cfg)

elf_train_cfg = dx.TrainingConfig(
    lr=1e-3,
    epochs=1,             # raise for real runs — 1 epoch is just a smoke test
    batch_size=16,
    tokenizer_type="t5",  # ELF requires T5 token IDs for the frozen encoder
    val_frac=0.1,
    eval_iters=10,
)

elf_run = dx.Trainer(elf_paradigm, elf_train_cfg).fit("tiny_shakespeare.txt")
print("Checkpoint saved to:", elf_run)

In [ ]:
# ── Generate — ELF denoises from pure Gaussian noise (unconditional) ─────────
import jax

from dantinox.utils.tokenizer import load_tokenizer_from_file

elf_model = dx.load(elf_run, paradigm=elf_paradigm)
tokens = elf_paradigm.generate(
    elf_model, None, jax.random.PRNGKey(0),
    gen_len=64,      # sequence length to synthesise
    n_steps=32,      # more steps → higher quality, slower
    cfg_scale=1.5,
)

tok = load_tokenizer_from_file(f"{elf_run}/tokenizer.json")
print(tok.decode([int(t) for t in tokens[0]]))

---
## 8 — HuggingFace Hub *(optional)*

Push your checkpoint so anyone can load it with `Generator("your-name/your-model")`.

You need a [HuggingFace account](https://huggingface.co/join) and a write token.

In [ ]:
HF_TOKEN  = "hf_..."          # paste your HF token here
HF_REPO   = "your-name/dantinox-dante"   # repo to create / update

from dantinox import push

url = push(run_dir, HF_REPO, token=HF_TOKEN, private=False)
print(f"Pushed to: {url}")

In [ ]:
# Load directly from the Hub on any machine — no pull step needed
gen_hub = Generator(HF_REPO, token=HF_TOKEN)
print(gen_hub.generate("Nel mezzo del cammin ", max_new_tokens=100))

---
## 9 — LoRA Fine-Tuning *(optional)*

Fine-tune with Low-Rank Adapters — only ~0.2 % of parameters are trained while the base weights stay frozen.

| Parameter | What it controls |
|---|---|
| `lora_rank` | adapter bottleneck width — higher = more capacity but more memory |
| `lora_alpha` | scales the adapter output: effective LR = lr × alpha / rank |
| `lora_dropout` | dropout applied inside the adapter (regularisation) |
| `lora_targets` | which modules to adapt: `"attention"` \| `"mlp"` \| `"all"` |

In [ ]:
import dataclasses

import dantinox as dx

# Same architecture as the base run, with LoRA adapters switched on
lora_model_cfg = dataclasses.replace(
    paradigm.config,
    use_lora=True,
    lora_rank=8,            # rank 4–16 covers most use cases
    lora_alpha=16.0,        # keep alpha == 2 * rank as a starting point
    lora_dropout=0.05,      # small dropout helps with short fine-tuning corpora
    lora_targets="attention",  # "attention" | "ffn" | "all"
)

# Fine-tuning schedule — shorter run, lower LR than pre-training
ft_cfg = dataclasses.replace(
    train_cfg,
    epochs=50,
    lr=1e-4,                # 3–10× lower than pre-training LR
    lr_schedule="cosine",
    warmup_steps=20,
    patience=5,
    # dataset_name="your-name/your-corpus",  # swap dataset to adapt domains
)

ft_run = dx.Trainer(dx.ARParadigm(lora_model_cfg), ft_cfg).fit()
print(f"\nLoRA checkpoint: {ft_run}")

# Generate from the fine-tuned model
from dantinox import Generator
ft_gen = Generator(ft_run, seed=42)
print(ft_gen.generate("Nel mezzo del cammin ", max_new_tokens=200, temperature=0.8))

---
## 10 — Benchmark

`BenchmarkRunner` measures:
- **Prefill latency** — time (ms) to process the prompt in a single forward pass
- **Decode throughput** — tokens/second at sequence lengths `[64, 128, 256]` (BS=1)
- **Batch throughput** — tokens/second at batch sizes `[1, 4, 16, 64]` (seq=256)
- **FLOPs** — XLA cost analysis for both prefill and decode steps
- **Theoretical KV-cache size** — bytes per layer at `max_context`

Results are saved to `benchmark_results.csv` and returned as a DataFrame.

In [ ]:
import os

from dantinox import BenchmarkRunner

# Keep seq-lens and batch sizes small for Colab's T4 (16 GB VRAM)
runner = BenchmarkRunner(
    "runs",
    seq_lens=[64, 128, 256],
    batch_sizes=[1, 4, 16, 64],
)

# Benchmark only the run we just trained (pass the folder name, not the full path)
run_name = os.path.basename(run_dir)
df = runner.run(run_names=[run_name], out_csv="benchmark_results.csv")

# Pretty-print the key columns
cols = [
    "run", "type", "params_m", "theoretical_cache_mb",
    "prefill_ms", "tps_64", "tps_128", "tps_256",
    "tps_bs1", "tps_bs4", "tps_bs16", "tps_bs64",
    "decode_gflops", "val_loss",
]
display(df[[c for c in cols if c in df.columns]].T)

In [ ]:
# ── Benchmark ALL runs in the runs/ directory ─────────────────────
# Useful if you trained multiple configs (e.g., after LoRA fine-tuning)
df_all = runner.run(out_csv="benchmark_results_all.csv")
print(f"Benchmarked {len(df_all)} run(s)")
display(df_all[["run", "type", "params_m", "prefill_ms", "tps_256", "val_loss"]])

---
## 11 — Plots

`Plotter` generates charts from the benchmark CSV — no extra setup needed.

| Group | What it shows |
|---|---|
| `perf` | Decode throughput vs seq-len · Batch throughput vs batch-size · Prefill latency |
| `3d` | 3-D surface: throughput × seq-len × batch-size |
| `insights` | Quality vs throughput · Cache size vs throughput (meaningful with ≥2 runs) |

In [ ]:
from dantinox import Plotter

saved = Plotter("benchmark_results.csv", out_dir="plots").run()
for group, path in saved.items():
    print(f"{group}: {path}")

In [ ]:
# Display all saved PNGs inline
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

for group, path in saved.items():
    img = mpimg.imread(path)
    fig, ax = plt.subplots(figsize=(14, 5) if group == "perf" else (8, 6))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(group)
    plt.tight_layout()
    plt.show()

---
## Tips

### A100 large model (GQA)

```python
model_cfg = dx.ModelConfig(
    dim=512, n_heads=16, head_size=32, num_blocks=12,
    max_context=512, vocab_size=256,
    attention="gqa", kv_heads=4,
    use_swiglu=True, norm="rmsnorm", use_flash=True,
    weight_tying=True, gradient_checkpointing=True, dropout=0.1,
)
train_cfg = dx.TrainingConfig(
    optimizer="adamw", lr=3e-4, lr_schedule="wsd",
    warmup_steps=500, grad_clip=1.0, use_bf16=True,
    epochs=1000, batch_size=128, grad_accum=8, patience=20, seed=42,
    dataset_source="huggingface", dataset_name="Daniele/dante-corpus",
    dataset_text_field="text", streaming=True, tokenizer_type="char",
)
run_dir = dx.Trainer(dx.ARParadigm(model_cfg), train_cfg).fit()
```

### Discrete diffusion (LLaDA-style) instead of AR

```python
paradigm = dx.build("discrete", dim=256, n_heads=8, head_size=32,
                    num_blocks=6, max_context=256, vocab_size=256,
                    noise_schedule="cosine", mask_token_id=4)
run_dir  = dx.Trainer(paradigm, train_cfg).fit()
```

### One-liner training

```python
run_dir = dx.fit("ar", "corpus.txt", dim=256, n_heads=8, head_size=32,
                 num_blocks=6, vocab_size=256, lr=3e-4, epochs=10)
print(dx.quick_generate(run_dir, "Once upon a time"))
```

### Out of memory on T4?

- lower `batch_size` (and raise `grad_accum` to keep the effective batch)
- set `gradient_checkpointing=True`
- set `use_bf16=True`
- reduce `max_context` or `num_blocks`